# Regime-Change Simulation

Shows how the PD-AMM tracks a StepOracle through a regime change, with informed traders and funding pressure realigning the market.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))

import math
import numpy as np
import matplotlib.pyplot as plt

from pdamm.amm.gaussian_mixture import GaussianMixtureAMM, GaussianMixtureParams
from pdamm.amm.anchor import CompositeAnchor, OracleAnchor
from pdamm.amm.oracle import StepOracle, RandomWalkOracle
from pdamm.amm.funding import FundingConfig
from pdamm.amm.perp_market import PerpMarket
from pdamm.agents.lp_provider import LPProvider
from pdamm.agents.informed_trader import InformedTrader
from pdamm.agents.noise_trader import NoiseTrader

K_RUST = 5.0 * math.sqrt(10.0 * math.sqrt(math.pi))


## Step-function regime change

In [ ]:
oracle = StepOracle(initial_mu=95.0, steps=[(100, 120.0), (200, 95.0)])
anchor = CompositeAnchor(sub_anchors=[OracleAnchor(oracle_feed=oracle)])
amm = GaussianMixtureAMM(
    b=50.0, k=K_RUST,
    params=GaussianMixtureParams.single(mu=95.0, sigma=10.0),
    max_collateral_per_trade=50.0,
)
config = FundingConfig(funding_rate_bps=200.0, slots_per_period=10, kl_cap=10.0, min_kl_threshold=0.001)
market = PerpMarket(amm=amm, anchor=anchor, funding_config=config, funding_interval=10)

lp = LPProvider("lp0", initial_deposit=150.0)
lp.enter(market)

noise = NoiseTrader("noise", mu_shock_scale=3.0, seed=7)

slot_hist, oracle_hist, amm_hist, kl_hist = [], [], [], []
for step in range(300):
    obs_mu = oracle.observe(market.slot).mu
    informed = InformedTrader(
        f"i_{step}",
        GaussianMixtureParams.single(mu=obs_mu, sigma=9.0),
        step_fraction=0.15, max_fraction_of_b=0.10,
    )
    informed.step(market)
    noise.step(market)
    market.tick(1)
    slot_hist.append(market.slot)
    oracle_hist.append(oracle.observe(market.slot).mu)
    amm_hist.append(market.amm.current_mu)
    kl_hist.append(market.current_kl_from_anchor())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(slot_hist, oracle_hist, "r--", label="Oracle mu", alpha=0.8)
ax1.plot(slot_hist, amm_hist, "b-", label="AMM mu")
ax1.axvline(100, color="gray", linestyle=":", alpha=0.7, label="Regime change")
ax1.axvline(200, color="gray", linestyle=":", alpha=0.7)
ax1.set_ylabel("Distribution mean")
ax1.set_title("AMM tracking a step-function oracle through regime changes")
ax1.legend()

ax2.plot(slot_hist, kl_hist, color="purple")
ax2.axvline(100, color="gray", linestyle=":", alpha=0.7)
ax2.axvline(200, color="gray", linestyle=":", alpha=0.7)
ax2.set_ylabel("KL(AMM || anchor)")
ax2.set_xlabel("Slot")
ax2.set_title("KL divergence (funding pressure)")
plt.tight_layout()
plt.show()

tracking_err = abs(amm_hist[-1] - oracle_hist[-1])
print(f"Final tracking error: {tracking_err:.2f}")
print(f"LP NAV: {market.lp_nav('lp0'):.3f}  (deposited: 150.0)")


## Random-walk oracle: long-run tracking

In [ ]:
oracle2 = RandomWalkOracle(
    initial_mu=100.0, vol=0.4,
    mean_reversion=0.05, long_run_mu=100.0, seed=42,
)
anchor2 = CompositeAnchor(sub_anchors=[OracleAnchor(oracle_feed=oracle2)])
amm2 = GaussianMixtureAMM(
    b=50.0, k=K_RUST,
    params=GaussianMixtureParams.single(mu=100.0, sigma=10.0),
    max_collateral_per_trade=50.0,
)
config2 = FundingConfig(funding_rate_bps=200.0, slots_per_period=10, kl_cap=10.0, min_kl_threshold=0.001)
market2 = PerpMarket(amm=amm2, anchor=anchor2, funding_config=config2, funding_interval=10)

lp2 = LPProvider("lp0", initial_deposit=150.0)
lp2.enter(market2)

slots2, oracle2_hist, amm2_hist, err_hist = [], [], [], []
for step in range(500):
    obs_mu = oracle2.observe(market2.slot).mu
    informed2 = InformedTrader(
        f"i_{step}",
        GaussianMixtureParams.single(mu=obs_mu, sigma=9.0),
        step_fraction=0.12, max_fraction_of_b=0.08,
    )
    informed2.step(market2)
    market2.tick(1)
    slots2.append(market2.slot)
    o_mu = oracle2.observe(market2.slot).mu
    oracle2_hist.append(o_mu)
    amm2_hist.append(market2.amm.current_mu)
    err_hist.append(abs(market2.amm.current_mu - o_mu))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(slots2, oracle2_hist, "r--", label="Oracle", alpha=0.8)
ax1.plot(slots2, amm2_hist, "b-", label="AMM mu", alpha=0.9)
ax1.set_ylabel("mu")
ax1.set_title("Random-walk oracle tracking (500 slots)")
ax1.legend()

ax2.plot(slots2, err_hist, color="orange")
ax2.set_ylabel("|AMM - oracle|")
ax2.set_xlabel("Slot")
ax2.set_title("Tracking error")
plt.tight_layout()
plt.show()

avg_err = sum(err_hist[-100:]) / 100
print(f"Average tracking error (last 100 slots): {avg_err:.3f}")
print(f"LP NAV: {market2.lp_nav('lp0'):.3f}")
